# Automatic Speech Recognition with Classical Language Models

Author: Vladyslav Cherniuk

### 1. Training n-grams

I used KenLM for scoring selecting sentences. However, KenLM doesn't allow creating a 1-gram model. It throws an error "This ngram implementation assumes at least a bigram model." when trying to create and arpa model for 1-gram. So I tried 2-gram, 3-gram and 4-gram instead.

I've downloaded the [Wikipedia base dataset](https://lang.org.ua/en/ubertext/) and divided into 3 different datasets of different sizes:
- full ("ubertext.wikipedia.filter_rus_gcld+short.text_only.txt", 4.6 GB)
- half ("ubertext.wikipedia.filter_rus_gcld+short.text_only.half.txt", 2.3 GB)
- quarter ("ubertext.wikipedia.filter_rus_gcld+short.text_only.quarter.txt", 1.2 GB)

In [1]:
!wget -O - https://kheafield.com/code/kenlm.tar.gz | tar xz
!mkdir kenlm/build && cd kenlm/build && cmake .. && make -j2
!ls kenlm/build/bin

In [2]:
!kenlm/build/bin/lmplz -o 2 <"data/ubertext.wikipedia.filter_rus_gcld+short.text_only.txt" > "models/2gram.arpa"
!kenlm/build/bin/build_binary models/2gram.arpa models/2gram.binary
!rm models/2gram.arpa

In [3]:
!kenlm/build/bin/lmplz -o 3 <"data/ubertext.wikipedia.filter_rus_gcld+short.text_only.txt" > "models/3gram.arpa"
!kenlm/build/bin/build_binary models/3gram.arpa models/3gram.binary
!rm models/3gram.arpa

In [4]:
!kenlm/build/bin/lmplz -o 4 <"data/ubertext.wikipedia.filter_rus_gcld+short.text_only.txt" > "models/4gram.arpa"
!kenlm/build/bin/build_binary models/4gram.arpa models/4gram.binary
!rm models/4gram.arpa

In [5]:
!kenlm/build/bin/lmplz -o 2 <"data/ubertext.wikipedia.filter_rus_gcld+short.text_only.half.txt" > "models/2gram_half.arpa"
!kenlm/build/bin/build_binary models/2gram_half.arpa models/2gram_half.binary
!rm models/2gram_half.arpa

In [6]:
!kenlm/build/bin/lmplz -o 3 <"data/ubertext.wikipedia.filter_rus_gcld+short.text_only.half.txt" > "models/3gram_half.arpa"
!kenlm/build/bin/build_binary models/3gram_half.arpa models/3gram_half.binary
!rm models/3gram_half.arpa

In [7]:
!kenlm/build/bin/lmplz -o 4 <"data/ubertext.wikipedia.filter_rus_gcld+short.text_only.half.txt" > "models/4gram_half.arpa"
!kenlm/build/bin/build_binary models/4gram_half.arpa models/4gram_half.binary
!rm models/4gram_half.arpa

In [8]:
!kenlm/build/bin/lmplz -o 2 <"data/ubertext.wikipedia.filter_rus_gcld+short.text_only.quarter.txt" > "models/2gram_quarter.arpa"
!kenlm/build/bin/build_binary models/2gram_quarter.arpa models/2gram_quarter.binary
!rm models/2gram_quarter.arpa

In [9]:
!kenlm/build/bin/lmplz -o 3 <"data/ubertext.wikipedia.filter_rus_gcld+short.text_only.quarter.txt" > "models/3gram_quarter.arpa"
!kenlm/build/bin/build_binary models/3gram_quarter.arpa models/3gram_quarter.binary
!rm models/3gram_quarter.arpa

In [10]:
!kenlm/build/bin/lmplz -o 4 <"data/ubertext.wikipedia.filter_rus_gcld+short.text_only.quarter.txt" > "models/4gram_quarter.arpa"
!kenlm/build/bin/build_binary models/4gram_quarter.arpa models/4gram_quarter.binary
!rm models/4gram_quarter.arpa

In [11]:
%pip install "Cython>=3.0.12,<4"

import subprocess
import sys

# Regenerate the bundled Cython wrapper for the current Python version.
subprocess.run(
    [sys.executable, "-m", "cython", "--cplus", "-3", "kenlm/python/kenlm.pyx"],
    check=True,
)
%pip install ./kenlm


[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
Processing ./kenlm
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for kenlm: filename=kenlm-0.0.0-cp313-cp313-linux_x86_64.whl size=2966463 sha256=50e7463e4df17f8ccfe551b0db099ba9a39b7b57d3c67d55fc2d727d5998c66d
  Stored in directory: /tmp/pip-ephem-wheel-cache-opq_zlwy/wheels/3f/f3/61/4c13983791c79b3ad17f358ac0b3e16d276787f64d52b92dd1
Successfully built kenlm
  Attempting uninstall: kenlm
    Found existing installation: kenlm 0.0.0
    Uninstalling kenlm-0.0.0:
      Successfully uninstalled kenlm-0.0.0

[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


### 2. Evaluating models



I used KenLM Model's `perplexity()` method as a score to compare the sentences, where the one with the lowest score is selected.

In [18]:
import kenlm
import json
import math
import pandas as pd

pd.set_option('display.max_colwidth', None)

models = {}

for n in range(2, 5):
    row = {}
    for suffix in ['', '_half', '_quarter']:
        row[f"model{suffix}"] = kenlm.Model(f"models/{n}gram{suffix}.binary")

    models[f"{n}gram"] = row

models_df = pd.DataFrame(models, index = models.keys())

In [23]:
models  = pd.DataFrame(models, index = models['2gram'].keys())

In [24]:
tests = []

with open('data/ua_asr_hypotheses_500.jsonl') as f:
    lines = f.readlines()
    for line in lines:
        tests.append(json.loads(line))

def get_predictions(tests, model):
    predictions = []

    for test in tests:
        min_score = math.inf
        min_index = 0
        for index, hypothesis in enumerate(test['hypotheses']):
            score = model.perplexity(hypothesis)
            if score < min_score:
                min_score = score
                min_index = index

        predictions.append({'reference': test['reference'], 'hypothesis': test['hypotheses'][min_index]})

    return predictions

def get_accuracy(predictions):
    return (predictions['hypothesis'] == predictions['reference']).sum() / len(predictions)

def get_incorrect(predictions):
    return predictions[predictions['hypothesis'] != predictions['reference']]

predictions = {}
wrong_predictions = {}
accuracies = {}

for col_index in models.columns:
    row_accuracies = {}
    row_predictions = {}
    row_wrong_predictions = {}
    for row_index, row in models.iterrows():
        model_predictions = get_predictions(tests, models.loc[row_index, col_index])
        df = pd.DataFrame(model_predictions)
        row_predictions[row_index] = df
        row_wrong_predictions[row_index] = get_incorrect(df)
        row_accuracies[row_index] = get_accuracy(df)

    predictions[col_index] = row_predictions
    wrong_predictions[col_index] = row_wrong_predictions
    accuracies[col_index] = row_accuracies

display(pd.DataFrame(accuracies))

,2gram,3gram,4gram
model,0.734,0.730,0.732
model_half,0.740,0.742,0.738
model_quarter,0.724,0.726,0.728


Here we can see that all models have relatively close accuracy, but the best one is for 3-gram model that was trained on half the dataset.

Predictions that differ for 2-gram trained on full dataset, half the dataset and its quarter:

In [25]:
rows = {}
column_names = []

for index in wrong_predictions['2gram'].keys():
    rows['reference'] = wrong_predictions['2gram'][index]['reference']
    column_name = f"{index}_hypothesis"
    rows[column_name] = wrong_predictions['2gram'][index]['hypothesis']
    column_names.append(column_name)

wrong_2gram_predictions = pd.DataFrame(rows)
wrong_2gram_predictions = wrong_2gram_predictions[wrong_2gram_predictions[column_names].nunique(axis = 1) > 1]

display(wrong_2gram_predictions)


,reference,model_hypothesis,model_half_hypothesis,model_quarter_hypothesis
62,Донні ВАН ДЕ БЕК: «Аякс повинен насолодитися цим матчем»,Данні ВАН ДЕ БЕК: « Аякс повинен насолодитися цим матчем »,Донні ВАН ДЕ БЕК: « Аякс повинен насолодитися цим матчем »,Донні ВАН ДЕ БЕК: « Аякс повинен насолодитися цим матчем »
113,"Сенцов розповів, куди саме його етапували","Синцов розповів, куди саме його етапували","Сенцов розповів, куди саме його його етапували","Синцов розповів, куди саме його етапували"
153,Ювентус – Удінезе – 4:1. Відео голів та огляд матчу,Ювентус – Удінезе – 4: 1. Відео голів та огляд матчу,Ювентус – Удінезе – 4: 1. Відео голів та огляд матчу,Ювентус – Удинезе – 4: 1. Відео голів та огляд матчу
159,"""Свобода"" висунула заарештованого Сиротюка першим номером на виборах до Київради",""" Свободо "" висунула заарештованого Сиротюка першим номером на виборах до Київради",""" Свобода "" висунула заарештованого Сиротюка першим номером на виборах до Київради",""" Свобода "" висунула заарештованого Сиротюка першим номером на виборах до Київради"
172,Володимир ПЯТЕНКО: «У Зорі мало шансів»,Володимир ПЯТЕНГО: « У Зорі мало шансів »,Володимир Володимир ПЯТЕНКО: « У Зорі мало шансів »,Володимир Володимир ПЯТЕНКО: « У Зорі мало шансів »
183,"Російські військові дезертирують з ""ДНР"" через затримки грошей, - ГУР","Російські військові дезертирують з "" ДНР "" через затримки грошей, - ГУР","Російські військові десердирують з "" ДНР "" через затримки грошей, - ГУР","Російські військові десердирують з "" ДНР "" через затримки грошей, - ГУР"
193,«Не мойєсовський нападник». Чому Вест Хем продав Себастьяна Аллера,« мойєсовський нападник ». Чому Вест Хем продав Себастьяна Аллера,« мойєсовський нападник ». Чому Вест Хем продав Себастьяна Аллера,« Не мойєсовський нападник ». Чому Вест Хем продав Себостьяна Аллера
247,"""Мене активно намагаються зняти з посади"", - Труба про справу щодо хабаря Щербині",""" Мене активно намагаються зняти з посади "", - Труба про справу щодо хабаря Щербині",""" Мене активно намагаються зняти з посади "", - Труба про справу щодо хабаря Щербині",""" Мене активно намагаються зняте з посади "", - Труба про справу щодо хабаря Щербині"
249,Дерек ЧІСОРА: «Хочу ще один бій у цьому році. Суперники - Усик або Паркер»,Дерек ЧІСОРО: « Хочу ще один бій у цьому році. Суперники - Усік або Паркер »,Дерек ЧІСОРО: « Хочу ще один бій у цьому році. Суперники - Усік або Паркер »,Дерек ЧІСОРА: « Хочу ще один бій у цьому році. Супернеки - Усик або Паркер »
275,Світовий банк виділить Україні кредит на $300 млн для боротьби з COVID-19,Світовий банк виділить Україні кредит на $ 300 млн для боротьби з COVID - 19,Світовий банк виділить Україні кредит на $ 300 млн для боротьби з з COVID - 19,Світовий банк виділить Україні кредит на $ 300 млн для боротьби з з COVID - 19


NaN here means that there was a correct prediction.

So we can see that the models trained on smaller datasets have more general grammatic errors: "десертирують" (183), "зняте" (247), "з з" (275), "вигубовувати" (280), "дітьме" and "що б" (311), "Про Про" (456), "кабочком" (496). However, the model trained on full dataset has more errors in names: "Данні ВАН ДЕ БЕК" (62), "Синцов" (113), "Свободо" (159), "ПЯТЕНГО" (172), "ЧІСОРО" (249) - I think, because they just take bigger portion of the whole file (half or quarter of the corpus).

2-gram wrong predictions:

In [26]:
display(wrong_predictions['2gram']['model'].iloc[0:60])

,reference,hypothesis
6,"Магда: Парламент оновлять політичні партії-""бройлери""","Магда: Парламент оновлять політичні партії - "" бройлери """
13,"Коломойський вимагає повернути йому не тільки $2 млрд, а й акції Приватбанку","Коломойський вимагає повернути йому не тільки $ 2 млрд, а й акції Приватбанку"
16,ОФІЦІЙНО: Дніпро-1 підписав Кобахідзе,ОФІЦІЙНО: Дніпро - 1 підписав Кобахідзе
22,"У Держдумі Росії марять таборами ""ІД"" на Херсонщині","У Держдумі Росії марять таборами "" ІД "" на Херсонщині"
25,СТАХОВСЬКИЙ: «Матч Ференцвароша в літаку додивлявся. Ребров – герой»,СТАХОВСЬКИЙ: « Матч Ференцвароша в в літаку дотивлявся. Ребров – герой »
32,"Мірча ЛУЧЕСКУ: «Ми не розслабилися, коли виграли чемпіонат України»","Мірча ЛУЧЕСКУ: « Ми ж розслабилися, коли виграли чемпіонат України »"
39,Російські хакери викрали дані у підрядника Apple і вимагають викуп у $50 мільйонів,Російські хакери викрали дані у підрядника Apple і вимагають викуп у $ 50 мільйонів
48,"Єрмак пропонує прийняти закон, що каратиме за публікацію ""прослушки"" топ-чиновників та політиків","Єрмак пропонує прийняти закон, що каратиме за публікацію "" прослушки "" топ - чиновників та політиків"
49,Атлетіко обіграв Бетіс на виїзді,Атлетіко обіграв Бедіс на виїзді
51,"Країни ОПЕК+ готові знизити обсяг видобутку нафти, не погодилась тільки Мексика","Країни ОПЕК + готові знизити обсяг видобутку нафти, погодилась тільки Мексика"


We can see that most wrong predictions contain specific symbols surrounded by spaces. I will add preprocessing for the sentences to try improving the scores. As correct references mostly have such symbols stuck to words, I think it might be useful to delete the spaces around the symbols which are surrounded by them. And also to remove extra spaces inside the bracket pairs (`" one two "` => `"one two"`) This way, when calculating perplexity, we will achieve the same perplexity score for sentences that differ just by the number of spaces. Then, to select the best one, we'll select the one that has fewer spaces ("words").

For example:

In [64]:
import re

init = 'Магда: Парламент оновлять політичні партії - " бройлери "'

result = re.sub(
    r'(["\'])\s*(.*?)\s*\1|«\s*(.*?)\s*»',
    lambda m: f'{m.group(1)}{m.group(2)}{m.group(1)}'
    if m.group(1)
    else f'«{m.group(3)}»',
    init,
)
result = re.sub(r"\s*([-'])\s*", r'\1', result)

print(f"Initial sentence: {init}")
print(f"Score: {models.loc['model', '2gram'].perplexity(init)}")
print(f"Resulting sentence: {result}")
print(f"Score: {models.loc['model', '2gram'].perplexity(result)}")

Initial sentence: Магда: Парламент оновлять політичні партії - " бройлери "
Score: 66081.3123096234
Resulting sentence: Магда: Парламент оновлять політичні партії-"бройлери"
Score: 1289126.7300835946


In [67]:
import re

def get_predictions_with_cleaned_hypotheses(tests, model):
    predictions = []

    for test in tests:
        min_score = math.inf
        min_index = 0
        min_length = math.inf
        for index, hypothesis in enumerate(test['hypotheses']):
            length = len(hypothesis.split())
            hypothesis = re.sub(
                r'(["\'])\s*(.*?)\s*\1|«\s*(.*?)\s*»',
                lambda m: f'{m.group(1)}{m.group(2)}{m.group(1)}'
                if m.group(1)
                else f'«{m.group(3)}»',
                hypothesis,
            )
            hypothesis = re.sub(r"\s*([-'])\s*", r'\1', hypothesis)
            score = model.perplexity(hypothesis)
            if score <= min_score and length < min_length:
                min_score = score
                min_index = index
                min_length = length

        predictions.append({'reference': test['reference'], 'hypothesis': test['hypotheses'][min_index]})

    return predictions


predictions = {}
wrong_predictions = {}
accuracies = {}

for col_index in models.columns:
    row_accuracies = {}
    row_predictions = {}
    row_wrong_predictions = {}
    for row_index, row in models.iterrows():
        model_predictions = get_predictions_with_cleaned_hypotheses(tests, models.loc[row_index, col_index])
        df = pd.DataFrame(model_predictions)
        row_predictions[row_index] = df
        row_wrong_predictions[row_index] = get_incorrect(df)
        row_accuracies[row_index] = get_accuracy(df)

    predictions[col_index] = row_predictions
    wrong_predictions[col_index] = row_wrong_predictions
    accuracies[col_index] = row_accuracies

display(pd.DataFrame(accuracies))

,2gram,3gram,4gram
model,0.996,0.992,0.992
model_half,0.996,0.994,0.992
model_quarter,0.996,0.994,0.994


In [68]:
display(wrong_predictions['2gram']['model'].iloc[0:60])

,reference,hypothesis
60,Кривий логотип Apple на iPhone підвищив вартість смартфона в рази,Кривий логотип Apple iPhone підвищив вартість смартфона в рази
393,Парубій підписав закон про перехід громад у ПЦУ,Парубій підписав закон про перехід громад ПЦУ
